# 02 — Preparação da base de conhecimento

## Objetivo
Carregar as tabelas relacionais necessárias, limpar as avaliações, enriquecer seus metadados e transformá-las em documentos para o RAG.

A lógica reutilizada está em `src/data/loader.py` e `src/data/preprocessing.py`.

**Princípio de modelagem:** uma avaliação com comentário representa um documento semântico. As tabelas de itens, produtos, vendedores e pagamentos são agregadas por pedido para não multiplicar artificialmente as avaliações.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
from src.data.loader import load_olist_raw
from src.data.preprocessing import prepare_reviews, enrich_reviews, build_documents

raw = load_olist_raw(RAW_DIR)
print("Tabelas carregadas:")
for name, frame in raw.items():
    print(f"- {name}: {frame.shape[0]:,} linhas x {frame.shape[1]} colunas")


## 1. Qualidade da tabela de avaliações

Compare a tabela original com o conjunto após limpeza textual.

In [ ]:
reviews_raw = raw["reviews"]
reviews_clean = prepare_reviews(reviews_raw)

print(f"Avaliações originais: {len(reviews_raw):,}")
print(f"Avaliações com comentário válido: {len(reviews_clean):,}")
print(f"Registros removidos por comentário vazio: {len(reviews_raw) - len(reviews_clean):,}")
display(reviews_clean[["document_id", "review_score", "review_creation_date", "review_comment_message"]].head())


## 2. Enriquecimento relacional

O enriquecimento associa informações de pedido, cliente, itens, produto/categoria, vendedor e pagamento. Relações com múltiplos itens são agregadas por pedido.

In [ ]:
prepared = enrich_reviews(raw)

print(f"Registros preparados: {len(prepared):,}")
print(f"IDs de avaliação distintos: {prepared['document_id'].nunique():,}")
display(prepared.head())


## 3. Campos e valores ausentes após enriquecimento

Nem toda avaliação possui correspondência em todas as dimensões. Os valores ausentes são preservados como metadados nulos, sem inventar informação.

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "tipo": prepared.dtypes.astype(str),
    "ausentes": prepared.isna().sum(),
    "ausentes_%": (prepared.isna().mean() * 100).round(2),
})
display(summary.sort_values("ausentes_%", ascending=False))


## 4. Datas e indicadores de entrega

Os campos derivados permitem contextualizar relatos de experiência de compra. Valores nulos podem ocorrer quando o pedido não foi entregue ou não possui datas suficientes.

In [ ]:
delivery_cols = [
    c for c in [
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date", "delivery_days", "delivery_delay_days"
    ] if c in prepared.columns
]
display(prepared[delivery_cols].describe(include="all").T)


## 5. Construção dos documentos

Cada documento contém identificador, texto original limpo e metadados serializáveis.

In [ ]:
documents = build_documents(prepared)

print(f"Documentos criados: {len(documents):,}")
display(documents[:3])
assert len(documents) == len(prepared)
assert len({doc["document_id"] for doc in documents}) == len(documents)


## 6. Persistência opcional da base preparada

O índice será reconstruído no próximo notebook. Este arquivo intermediário é útil para inspeção e auditoria; o pipeline oficial continua reproduzível a partir dos CSVs originais.

In [ ]:
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
prepared_file = PROCESSED_DIR / "reviews_enriched.csv"
prepared.to_csv(prepared_file, index=False, encoding="utf-8")
print(f"Base enriquecida salva em: {prepared_file.relative_to(ROOT)}")


## 7. Conclusões da preparação

Ao executar, verifique a quantidade de comentários válidos, a unicidade dos IDs, a cobertura dos metadados e a preservação de uma avaliação por documento.

**Próxima etapa:** [03 — Embeddings e indexação](03_embeddings_indexacao.ipynb).